In [ ]:
import pandas as pd
import numpy as np
from catboost import CatBoostRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')


# 1. YÜKLEME VE SÜTUN TEMİZLİĞİ
train = pd.read_csv('train.csv')
test = pd.read_csv('test_x.csv')

train.columns = train.columns.str.lower().str.replace(' ', '_').str.strip()
test.columns = test.columns.str.lower().str.replace(' ', '_').str.strip()
target_col = 'bilissel_performans_skoru'

# 2. AYNI GÜVENLİ ÖZELLİKLER
def create_safe_features(df):
    df = df.copy()
    if 'rem_yuzdesi' in df.columns and 'derin_uyku_yuzdesi' in df.columns:
        df['toplam_kaliteli_uyku'] = df['rem_yuzdesi'] + df['derin_uyku_yuzdesi']
    if 'uyku_oncesi_kafein_mg' in df.columns and 'uyku_oncesi_ekran_suresi_dk' in df.columns:
        df['uyku_bozucu_faktorler'] = (df['uyku_oncesi_kafein_mg']/100) + (df['uyku_oncesi_ekran_suresi_dk']/30)
    if 'gunluk_calisma_saati' in df.columns and 'stres_skoru' in df.columns:
        df['tukenmislik_riski'] = df['gunluk_calisma_saati'] * df['stres_skoru']
    return df

train = create_safe_features(train)
test = create_safe_features(test)

y = train[target_col]
X = train.drop(['id', target_col], axis=1)
X_test = test.drop(['id'], axis=1)

# 3. EKSİK VERİ DOLDURMA
num_cols = X.select_dtypes(include=[np.number]).columns
for col in num_cols:
    med_val = X[col].median()
    X[col] = X[col].fillna(med_val)
    X_test[col] = X_test[col].fillna(med_val)

cat_cols = X.select_dtypes(include=['object']).columns.tolist()
for col in cat_cols:
    X[col] = X[col].fillna('Missing').astype(str)
    X_test[col] = X_test[col].fillna('Missing').astype(str)

# 4. MULTI-SEED K-FOLD
SEEDS = [42, 123, 777]  # 3 Farklı evren
test_predictions = np.zeros(len(X_test))
oof_predictions = np.zeros(len(X))

print(f"Toplam 15 Model Eğitilecek ({len(SEEDS)} Seed x 5 Fold")

for seed in SEEDS:
    print(f"--- BAŞLIYOR: SEED {seed} ---")
    kf = KFold(n_splits=5, shuffle=True, random_state=seed)

    # Her seed için OOF takip dizisi
    seed_oof = np.zeros(len(X))

    for fold, (train_idx, val_idx) in enumerate(kf.split(X, y)):
        X_tr, y_tr = X.iloc[train_idx], y.iloc[train_idx]
        X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

        # Öğrenme oranını iyice kıstık (0.015), süreyi uzattık (2000)
        model = CatBoostRegressor(
            iterations=2000,
            learning_rate=0.015,
            depth=6,
            l2_leaf_reg=5,
            cat_features=cat_cols,
            random_seed=seed,
            verbose=0
        )

        model.fit(X_tr, y_tr, eval_set=(X_val, y_val), early_stopping_rounds=150)

        seed_oof[val_idx] = model.predict(X_val)
        # Final tahminlere (1 / 15) oranında ekleme yapıyoruz
        test_predictions += model.predict(X_test) / (5 * len(SEEDS))

    seed_rmse = np.sqrt(mean_squared_error(y, seed_oof))
    oof_predictions += seed_oof / len(SEEDS)
    print(f"Seed {seed} OOF RMSE: {seed_rmse:.4f}\n")

# 5. FİNAL SKOR
final_local_rmse = np.sqrt(mean_squared_error(y, oof_predictions))

print("="*60)
print(f"LOCAL RMSE: {final_local_rmse:.4f}")
print("="*60)

test_predictions = np.clip(test_predictions, 0, 10)
submission = pd.DataFrame({'id': test['id'], target_col: test_predictions})
submission.to_csv('Takim_141_version_6.csv', index=False)


Toplam 15 Model Eğitilecek (3 Seed x 5 Fold
--- BAŞLIYOR: SEED 42 ---
Seed 42 OOF RMSE: 1.2161

--- BAŞLIYOR: SEED 123 ---
Seed 123 OOF RMSE: 1.2158

--- BAŞLIYOR: SEED 777 ---
Seed 777 OOF RMSE: 1.2157

LOCAL RMSE: 1.2149
